This notebook uses altered neurotorch array and predictor classes that rely on the package tensorstore for indexing the data. This results in only single model iterations of the input and output arrays being stored in memory, as opposed to the entire array, saving immensely on memory usage.

In [7]:
import tensorstore as ts
import numpy as np
import ac_segmentation.neurotorch.datasets.dataset
import ac_segmentation.neurotorch.core.predictor
import ac_segmentation.neurotorch.nets.RSUNet

TensorPredictor = ac_segmentation.neurotorch.core.predictor.TensorPredictor
TensorArray = ac_segmentation.neurotorch.datasets.dataset.TensorArray

In [13]:
###Load input zarr 

data_in = ts.open({
         'driver':
             'zarr',
         'kvstore':
             'http://bigkahuna.corp.alleninstitute.org/ACdata/Users/kevin/ispim_ome_zarr/H17_x55_S39a_230808_highres/H17_x55_S39a_230808_highres.zarr/highres_Pos89/1/',
     # Use 100MB in-memory cache.
         'context': {
             'cache_pool': {
                 'total_bytes_limit': 100_000_000
             }
         },
         'recheck_cached_data':
         'open',
     })

data_in = await data_in
inarr = data_in[0,0,:,:,:].transpose()
inarr_shape = list(inarr.shape)[::-1]

In [14]:
###Create output zarr

out_path = 'file:///ACdata/Users/connorl/Example_OutArr.zarr'
data_out = ts.open({
     'driver': 'zarr',
     'kvstore': out_path,
 },
 dtype=ts.float32,
 chunk_layout=ts.ChunkLayout(chunk_shape=[64, 64, 64]),
 create=True,
 shape=inarr_shape).result()

outarr = data_out.transpose()

In [10]:
###Run segmentation (probability map is written directly to output zarr)

checkpt_file = "/home/russelt/some_ckpt.ckpt"
net = ac_segmentation.neurotorch.nets.RSUNet.RSUNet()
pred = TensorPredictor(net, checkpt_file, gpu_device=None)
in_arr = TensorArray(inarr)
out_arr = TensorArray(outarr)
%time pred.run(in_arr, out_arr, batch_size=80)

RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.